In [1]:
# Install requirements quietly using %pip (recommended in Jupyter)
try:
	get_ipython().run_line_magic('pip', 'install -q -r ../requirements.txt')
except Exception:
	# Fallback to shell command if %pip is not available
	import os
	os.system("pip install -q -r ../requirements.txt")

Note: you may need to restart the kernel to use updated packages.


In [2]:
# ========================================
# ENVIRONMENT VERIFICATION
# ========================================
import sys
import torch
import tarfile  # No need for try/except - this is standard library
import math
from pathlib import Path
from typing import Optional, Tuple, Callable, Dict

# Set environment variable for faster HF transfers
import os
os.environ.setdefault("HF_HUB_ENABLE_HF_TRANSFER", "1")

print(f"Python version: {sys.version}")
print(f"Python executable: {sys.executable}")
print(f"Virtual environment: {sys.prefix}")
print(f"✅ torch imported successfully! Version: {torch.__version__}")

Python version: 3.13.7 (main, Aug 14 2025, 11:12:11) [Clang 17.0.0 (clang-1700.0.13.3)]
Python executable: /Users/annedranowski/HQ/🚀 Projects/Knot Detector/.venv/bin/python
Virtual environment: /Users/annedranowski/HQ/🚀 Projects/Knot Detector/.venv
✅ torch imported successfully! Version: 2.8.0


In [3]:
# ========================================
# DEVICE CONFIGURATION
# ========================================
def pick_device():
    """Select best available device"""
    if torch.cuda.is_available():
        return "cuda"
    elif torch.backends.mps.is_available():
        return "mps"
    else:
        return "cpu"

DEVICE = pick_device()
print("Device:", DEVICE)

Device: mps


In [4]:
# ========================================
# DATA LOADING CONFIGURATION
# ========================================
# REDUNDANCY ISSUE: Multiple data loader definitions scattered throughout
# SOLUTION: Consolidate all data loading logic here

REPO_ID = "tr33hugg3r/knot-crossings"
HF_ZIP_FILENAME = "knot_crossings_dataset.tar.gz"  # Updated to match your actual file
REVISION = None
BATCH_SIZE = 32
NUM_WORKERS = 2 if not DEVICE.startswith("mps") else 0  # MPS doesn't support multiprocessing
IMG_SIZE = 480
USE_GRAYSCALE = True  # Changed to True since your model expects 1 channel
GRAYSCALE_CHANNELS = 1  # Model architecture expects 1 channel


In [5]:
# ========================================
# DATA TRANSFORMS
# ========================================
from torchvision.transforms import v2

# REDUNDANCY ISSUE: Transform definition scattered/duplicated
# SOLUTION: Single transform definition
data_transform = v2.Compose([
    v2.Grayscale(num_output_channels=GRAYSCALE_CHANNELS) if USE_GRAYSCALE else v2.Identity(),
    v2.Resize((IMG_SIZE, IMG_SIZE)),
    v2.ToImage(),
    v2.ToDtype(torch.float32, scale=True),
])

In [11]:
# Data Loading Utilities
import os, sys, zipfile, shutil, tempfile
from pathlib import Path
from typing import Optional, Tuple

os.environ.setdefault("HF_HUB_ENABLE_HF_TRANSFER", "1")  # speeds up HF downloads if hf_transfer is installed

import torch
from torch import nn
from torch.utils.data import DataLoader
from torchvision import datasets, transforms
from datasets import load_dataset, Image as HFImage

# ---- device & transforms ----
def pick_device() -> str:
    if torch.cuda.is_available(): return "cuda"
    if torch.backends.mps.is_available(): return "mps"
    return "cpu"

def build_transforms(img_size=480):
    return transforms.Compose([
        transforms.Resize((img_size, img_size)),
        transforms.ToTensor()
    ])

# ---- ImageFolder detection & loaders ----
def looks_like_imagefolder(root: Path) -> bool:
    train, test = root/"train", root/"test"
    if not (train.is_dir() and test.is_dir()): return False
    has_train_cls = any(p.is_dir() for p in train.iterdir())
    has_test_cls  = any(p.is_dir() for p in test.iterdir())
    return has_train_cls and has_test_cls

def build_loaders_from_imagefolder(data_root: Path, batch_size=32, num_workers=2, device="cpu"):
    tfs = build_transforms()
    train_ds = datasets.ImageFolder(data_root/"train", transform=tfs)
    test_ds  = datasets.ImageFolder(data_root/"test",  transform=tfs)
    pin = device.startswith("cuda")
    nw  = 0 if device.startswith("mps") else num_workers
    train_loader = DataLoader(train_ds, batch_size=batch_size, shuffle=True,
                              num_workers=nw, pin_memory=pin, persistent_workers=(nw>0))
    test_loader  = DataLoader(test_ds, batch_size=batch_size, shuffle=False,
                              num_workers=nw, pin_memory=pin, persistent_workers=(nw>0))
    return train_loader, test_loader, train_ds.classes

# ---- Zip safety + “find real dataset root” ----
def _is_within_directory(base: Path, target: Path) -> bool:
    try:
        target.resolve().relative_to(base.resolve())
        return True
    except ValueError:
        return False

def safe_extract(zip_path: Path, extract_to: Path) -> None:
    with zipfile.ZipFile(zip_path, "r") as zf:
        for m in zf.infolist():
            member_path = extract_to / m.filename
            if not _is_within_directory(extract_to, member_path):
                raise RuntimeError(f"Unsafe path in archive: {m.filename}")
        zf.extractall(extract_to)

def find_imagefolder_root(extract_root: Path) -> Path:
    """Return the dir (at or under extract_root) that has train/ and test/ with class subdirs."""
    if looks_like_imagefolder(extract_root):
        return extract_root
    # breadth-first: shallowest candidates first
    candidates = [p for p in extract_root.rglob("*") if p.is_dir()]
    candidates.sort(key=lambda p: len(p.relative_to(extract_root).parts))
    for p in candidates:
        if looks_like_imagefolder(p):
            return p
    raise FileNotFoundError("Could not find an ImageFolder (train/ + test/) in extracted archive.")

def extract_zip_to_imagefolder(zip_path: Path, dest_root: Path) -> Path:
    """
    Extract a dataset ZIP and normalize it so the result is:
        dest_root / {train,test}/<class>/
    Replaces dest_root if it exists. Returns dest_root.
    """
    dest_root = dest_root.resolve()
    if dest_root.exists():
        shutil.rmtree(dest_root)
    dest_root.mkdir(parents=True, exist_ok=True)

    with tempfile.TemporaryDirectory() as td:
        tmp = Path(td)
        safe_extract(zip_path, tmp)
        src_root = find_imagefolder_root(tmp)
        for sub in ("train", "test"):
            src = src_root / sub
            if src.is_dir():
                (dest_root / sub).mkdir(parents=True, exist_ok=True)
                for p in src.iterdir():
                    shutil.move(str(p), str(dest_root / sub / p.name))
    return dest_root

# ---- HF Parquet (fallback) ----
def build_datasets_from_hf_parquet(repo_id: str, revision: Optional[str]=None,
                                   image_key="image", label_key="label"):
    ds_train = load_dataset(repo_id, split="train", revision=revision)
    ds_test  = load_dataset(repo_id, split="test",  revision=revision)
    if not isinstance(ds_train.features[image_key], HFImage):
        ds_train = ds_train.cast_column(image_key, HFImage())
        ds_test  = ds_test.cast_column(image_key,  HFImage())
    tfs = build_transforms()
    def _transform(ex):
        ex[image_key] = tfs(ex[image_key])
        return ex
    ds_train.set_transform(_transform)
    ds_test.set_transform(_transform)
    feat = ds_train.features.get(label_key, None)
    class_names = getattr(feat, "names", None) if feat is not None else None
    return ds_train, ds_test, class_names

def build_loaders_from_hf_datasets(ds_train, ds_test, image_key="image", label_key="label",
                                   batch_size=32, shuffle=True):
    def collate(batch):
        xs = torch.stack([b[image_key] for b in batch])
        ys = torch.tensor([b[label_key] for b in batch], dtype=torch.long)
        return xs, ys
    train_loader = DataLoader(ds_train, batch_size=batch_size, shuffle=shuffle, collate_fn=collate)
    test_loader  = DataLoader(ds_test,  batch_size=batch_size, shuffle=False,   collate_fn=collate)
    if hasattr(ds_train.features[label_key], "names") and ds_train.features[label_key].names:
        class_names = list(ds_train.features[label_key].names)
    else:
        labels = sorted({int(r[label_key]) for r in ds_train})
        class_names = [str(i) for i in labels]
    return train_loader, test_loader, class_names


In [12]:
from huggingface_hub import snapshot_download, hf_hub_download
# DataResolver class to handle dataset resolution
class DataResolver:
    """
    Order:
      1) Use local repo data at data/raw/knot-crossings@main/ (fastest)
      2) Download and extract a single ZIP from HF (fast)
      3) Snapshot from HF (slower if many small files)
    """
    def __init__(self, repo_id: str, hf_zip_filename: str, revision: Optional[str]=None, repo_type: str="dataset"):
        self.repo_id = repo_id
        self.hf_zip_filename = hf_zip_filename  # e.g., "knot_images_480.zip"
        self.revision = revision
        self.repo_type = repo_type
        self.local_candidates = [
            Path("data/raw/knot-crossings@main"),
            Path("../data/raw/knot-crossings@main"),
        ]

    def _try_local(self) -> Optional[Path]:
        for root in self.local_candidates:
            root = root.resolve()
            if looks_like_imagefolder(root):
                print(f"✅ Using local dataset at: {root}")
                return root
        return None

    def _try_zip_from_hf(self, extract_to: Path) -> Optional[Path]:
        try:
            print(f"⬇️  Downloading ZIP from HF: {self.repo_id}/{self.hf_zip_filename}")
            zip_path = hf_hub_download(
                repo_id=self.repo_id,
                filename=self.hf_zip_filename,
                repo_type=self.repo_type,
                revision=self.revision
            )
            dest_root = Path(extract_to)
            root = extract_zip_to_imagefolder(Path(zip_path), dest_root)
            print(f"✅ Using HF ZIP (extracted) at: {root}")
            return root
        except Exception as e:
            print(f"⚠️ HF ZIP download failed: {e}")
            return None

    def _snapshot(self) -> Optional[Path]:
        try:
            print(f"⬇️  Falling back to snapshot_download for {self.repo_id} (may take longer)...")
            repo_path = snapshot_download(
                repo_id=self.repo_id,
                repo_type=self.repo_type,
                revision=self.revision,
                allow_patterns=["train/**", "test/**"]
            )
            root = Path(repo_path)
            if looks_like_imagefolder(root):
                print(f"✅ Using HF snapshot at: {root}")
                return root
            for p in root.iterdir():
                if p.is_dir() and looks_like_imagefolder(p):
                    print(f"✅ Using HF snapshot subdir at: {p}")
                    return p
            print("⚠️ Snapshot downloaded but train/test not found.")
            return None
        except Exception as e:
            print(f"❌ snapshot_download failed: {e}")
            return None

    def resolve(self, prefer_zip=True, extract_to=Path("data/raw/knot-crossings@main")) -> Tuple[Optional[Path], Optional[str]]:
        root = self._try_local()
        if root: return root, "local"
        if prefer_zip:
            root = self._try_zip_from_hf(Path(extract_to))
            if root: return root, "hf_zip"
        root = self._snapshot()
        if root: return root, "hf_snapshot"
        return None, None


In [13]:
# --- settings you can tweak at the top of the nb ---
REPO_ID = "tr33hugg3r/knot-crossings"   # HF dataset
HF_ZIP_FILENAME = "knot_crossings_dataset.tar.gz" # <-- set to your actual uploaded ZIP name
REVISION = None                         # optional: tag/commit
BATCH_SIZE = 32
NUM_WORKERS = 2

DEVICE = pick_device()
print("Device:", DEVICE)

# Try local/zip/snapshot in that order
resolver = DataResolver(repo_id=REPO_ID, hf_zip_filename=HF_ZIP_FILENAME, revision=REVISION)
data_root, source = resolver.resolve(prefer_zip=True, extract_to=Path("data/raw/knot-crossings@main"))

if data_root is not None:
    train_loader, test_loader, class_names = build_loaders_from_imagefolder(
        data_root, batch_size=BATCH_SIZE, num_workers=NUM_WORKERS, device=DEVICE
    )
    print(f"Data source: {source}")
else:
    print("ℹ️ No ImageFolder found; falling back to HF Parquet view.")
    ds_train, ds_test, _ = build_datasets_from_hf_parquet(REPO_ID, revision=REVISION)
    train_loader, test_loader, class_names = build_loaders_from_hf_datasets(
        ds_train, ds_test, batch_size=BATCH_SIZE
    )
    print("Data source: hf_parquet")

print("Classes:", class_names)
num_classes = len(class_names)


Device: mps
✅ Using local dataset at: /Users/annedranowski/HQ/🚀 Projects/Knot Detector/data/raw/knot-crossings@main
Data source: local
Classes: ['0', '10', '3', '4', '5', '6', '7', '8', '9']


In [14]:
# ========================================
# CLASS REMAPPING UTILITIES
# ========================================
# REDUNDANCY ISSUE: Multiple similar functions for class handling
# SOLUTION: Consolidated class handling

def numeric_sorted(names):
    """Sort class names numerically (10 comes after 9)"""
    return sorted(list(names), key=lambda s: int(s))

def normalize_imagefolder_classes(train_ds, test_ds, drop_test_only=False):
    """
    Normalize both datasets to use numeric class ordering
    and handle test-only classes consistently
    """
    # Get numeric ordering from training data
    train_sorted = numeric_sorted(train_ds.classes)
    
    # Update train dataset
    train_ds.classes = train_sorted
    train_ds.class_to_idx = {c: i for i, c in enumerate(train_sorted)}
    
    if drop_test_only:
        # Filter test dataset to only include training classes
        test_classes = [c for c in train_sorted if c in test_ds.classes]
        # Would need more implementation here...
        print(f"Dropping test-only classes: {set(test_ds.classes) - set(train_sorted)}")
    else:
        # Keep all test classes but sort numerically
        test_sorted = numeric_sorted(test_ds.classes)
        test_ds.classes = test_sorted
        test_ds.class_to_idx = {c: i for i, c in enumerate(test_sorted)}
        
        extra_in_test = set(test_ds.classes) - set(train_ds.classes)
        if extra_in_test:
            print(f"Warning: test has classes not in train: {numeric_sorted(extra_in_test)}")
    
    return train_sorted

# Apply class normalization
if 'train_loader' in locals():
    class_names = normalize_imagefolder_classes(train_loader.dataset, test_loader.dataset)
    num_classes = len(class_names)
    print("Final class order:", class_names)

Final class order: ['0', '3', '4', '5', '6', '7', '8', '9', '10']


In [15]:
# ========================================
# UTILITY FUNCTIONS
# ========================================
def describe_loader(loader, name="loader"):
    """Describe a DataLoader's properties"""
    bs = getattr(loader, "batch_size", None)
    n_batches = len(loader)
    ds = loader.dataset
    n_items = len(ds)
    classes = getattr(ds, "classes", None)
    print(f"{name}: {n_items} items | batches: {n_batches}" + (f" of {bs}" if bs else ""))
    if classes is not None:
        print(f"  classes ({len(classes)}): {classes}")

In [17]:
# REDUNDANCY ISSUE: Multiple helper function downloads
# TODO: SOLUTION: Single download with better error handling
def download_helper_functions():
    """Download helper functions if needed"""
    if Path("helper_functions.py").is_file():
        print("helper_functions.py already exists")
        return True
    
    try:
        print("Downloading helper_functions.py")
        import requests
        response = requests.get(
            "https://raw.githubusercontent.com/mrdbourke/pytorch-deep-learning/main/helper_functions.py"
        )
        response.raise_for_status()  # Raise exception for bad status codes
        
        with open("helper_functions.py", "wb") as f:
            f.write(response.content)
        return True
    except Exception as e:
        print(f"Failed to download helper_functions.py: {e}")
        return False

# Download helper functions
download_helper_functions()

True

In [18]:
# ========================================
# MODEL DEFINITION
# ========================================
from torch import nn

class KnotsModelCNN(nn.Module):
    """
    CNN for knot-crossing prediction.
    
    ISSUE: Constructor parameters don't match usage in loading section
    SOLUTION: Make parameters consistent with actual usage
    """
    def __init__(self, output_shape: int = 1, dropout: float = 0.7):
        super().__init__()
        
        # Feature extractor - expects 1-channel input for grayscale
        self.conv_1 = nn.Sequential(
            nn.Conv2d(1, 4, kernel_size=11, stride=1, dilation=2, padding=0),
            nn.BatchNorm2d(4),
            nn.ReLU(inplace=True),
            nn.MaxPool2d(2),
            
            nn.Conv2d(4, 16, kernel_size=5, stride=1, dilation=2, padding=0),
            nn.BatchNorm2d(16),
            nn.ReLU(inplace=True),
            nn.MaxPool2d(2),
            
            nn.Conv2d(16, 64, kernel_size=3, stride=1, dilation=2, padding=0),
            nn.BatchNorm2d(64),
            nn.ReLU(inplace=True),
            nn.MaxPool2d(2),
            
            nn.Conv2d(64, 256, kernel_size=3, stride=1, dilation=2, padding=0),
            nn.BatchNorm2d(256),
            nn.ReLU(inplace=True),
            nn.MaxPool2d(2),
            
            nn.Conv2d(256, 361, kernel_size=3, stride=1, padding=0),
            nn.BatchNorm2d(361),
            nn.ReLU(inplace=True),
            nn.MaxPool2d(2),
        )
        
        # Classifier
        # ISSUE: Very large FC layers (4096->4096) may be overkill
        # COMMENT: Consider smaller layers or Global Average Pooling
        self.classifier = nn.Sequential(
            nn.Flatten(),
            nn.LazyBatchNorm1d(),
            nn.Dropout(p=dropout),
            nn.LazyLinear(4096),  # Consider reducing this
            nn.BatchNorm1d(4096),
            nn.ReLU(inplace=True),
            nn.Dropout(p=dropout),
            nn.Linear(4096, 4096),  # This is very large
            nn.ReLU(inplace=True),
            nn.Linear(4096, output_shape),
        )
    
    def forward(self, x: torch.Tensor) -> torch.Tensor:
        x = self.conv_1(x)
        x = self.classifier(x)
        return x

In [19]:
# ========================================
# TRAINING UTILITIES
# ========================================
# REDUNDANCY ISSUE: train_step and test_step have duplicate code and inconsistent interfaces
# SOLUTION: Cleaner, more consistent training functions

def train_step(model, data_loader, loss_fn, optimizer, accuracy_fn, device=DEVICE):
    """Single training step"""
    model.train()
    train_loss, train_acc = 0, 0
    
    for X, y in data_loader:
        X, y = X.to(device), y.to(device)
        
        # Forward pass
        y_pred = model(X).squeeze(dim=1)
        
        # Calculate loss
        loss = loss_fn(y_pred, y.float())  # Ensure float for MSE
        train_loss += loss.item()
        train_acc += accuracy_fn(y_true=y, y_pred=y_pred.round())
        
        # Backward pass
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
    
    # Average metrics
    train_loss /= len(data_loader)
    train_acc /= len(data_loader)
    
    return train_loss, train_acc

def eval_step(model, data_loader, loss_fn, accuracy_fn, device=DEVICE):
    """Single evaluation step"""
    model.eval()
    eval_loss, eval_acc = 0, 0
    predictions, targets = [], []
    
    with torch.inference_mode():
        for X, y in data_loader:
            X, y = X.to(device), y.to(device)
            
            y_pred = model(X).squeeze(dim=1)
            
            eval_loss += loss_fn(y_pred, y.float()).item()
            eval_acc += accuracy_fn(y_true=y, y_pred=y_pred.round())
            
            # Store predictions for analysis
            predictions.extend(y_pred.round().cpu().numpy())
            targets.extend(y.cpu().numpy())
    
    eval_loss /= len(data_loader)
    eval_acc /= len(data_loader)
    
    return eval_loss, eval_acc, predictions, targets

In [20]:
# ========================================
# MODEL EVALUATION
# ========================================
# ISSUE: eval_model function is redundant with eval_step
# SOLUTION: Use eval_step consistently

def get_model_info(model):
    """Get model parameter count and size"""
    n_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
    param_bytes = sum(p.nelement() * p.element_size() for p in model.parameters())
    buffer_bytes = sum(b.nelement() * b.element_size() for b in model.buffers())
    size_mb = (param_bytes + buffer_bytes) / (1024 ** 2)
    
    return n_params, size_mb

In [21]:
# ========================================
# VISUALIZATION UTILITIES
# ========================================
import matplotlib.pyplot as plt

def plot_training_curves(train_losses, train_accs, val_losses, val_accs):
    """Plot training and validation curves"""
    epochs = range(1, len(train_losses) + 1)
    
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))
    
    # Loss curves
    ax1.plot(epochs, train_losses, label="Train Loss", linewidth=2)
    ax1.plot(epochs, val_losses, label="Validation Loss", linewidth=2)
    ax1.set_title("Training vs Validation Loss", fontsize=14)
    ax1.set_xlabel("Epoch", fontsize=12)
    ax1.set_ylabel("Loss", fontsize=12)
    ax1.legend()
    ax1.grid(True, linestyle="--", alpha=0.6)
    
    # Accuracy curves
    ax2.plot(epochs, train_accs, label="Train Accuracy", linewidth=2)
    ax2.plot(epochs, val_accs, label="Validation Accuracy", linewidth=2)
    ax2.set_title("Training vs Validation Accuracy", fontsize=14)
    ax2.set_xlabel("Epoch", fontsize=12)
    ax2.set_ylabel("Accuracy (%)", fontsize=12)
    ax2.legend()
    ax2.grid(True, linestyle="--", alpha=0.6)
    
    plt.tight_layout()
    plt.show()

In [25]:
# ========================================
# MAIN EXECUTION SECTION
# ========================================
# ISSUE: Multiple training loops with redundant code
# SOLUTION: Single, clean training loop

if __name__ == "__main__" and 'train_loader' in locals():
    # Import accuracy function
    try:
        from helper_functions import accuracy_fn
    except ImportError:
        print("Warning: Could not import accuracy_fn, define a basic one")
        def accuracy_fn(y_true, y_pred):
            return torch.eq(y_true, y_pred).sum().item() / len(y_pred) * 100
    
    # Model setup
    model = KnotsModelCNN(output_shape=1).to(DEVICE)
    # Initialize lazy layers with a dummy batch (batch size >= 2 for BatchNorm1d)
    dummy_batch = next(iter(train_loader))[0][:2].to(DEVICE)
    # If batch size < 2, duplicate the sample
    if dummy_batch.shape[0] < 2:
        dummy_batch = dummy_batch.repeat(2, 1, 1, 1)
    # Ensure dummy_batch has 1 channel for grayscale input
    if dummy_batch.shape[1] != 1:
        dummy_batch = dummy_batch.mean(dim=1, keepdim=True)
    _ = model(dummy_batch)
    n_params, size_mb = get_model_info(model)
    print(f"Model: {n_params:,} parameters, {size_mb:.2f} MB")
    
    # Training setup
    loss_fn = nn.MSELoss()
    optimizer = torch.optim.AdamW(model.parameters(), lr=3e-4, weight_decay=1e-4)
    scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
        optimizer, mode="min", factor=0.2, patience=2, threshold=1e-3
    )
    
    # Training loop
    EPOCHS = 20
    train_losses, train_accs = [], []
    val_losses, val_accs = [], []
    
    print("Starting training...")
    for epoch in range(EPOCHS):
        print(f"\nEpoch {epoch+1}/{EPOCHS}")
        
        # Train
        train_loss, train_acc = train_step(model, train_loader, loss_fn, optimizer, accuracy_fn)
        train_losses.append(train_loss)
        train_accs.append(train_acc)
        print(f"Train Loss: {train_loss:.4f}, Train Acc: {train_acc:.2f}%")
        
        # Validate
        val_loss, val_acc, _, _ = eval_step(model, test_loader, loss_fn, accuracy_fn)
        val_losses.append(val_loss)
        val_accs.append(val_acc)
        print(f"Val Loss: {val_loss:.4f}, Val Acc: {val_acc:.2f}%")
        
        # Learning rate scheduling
        scheduler.step(val_loss)
        
        # Early stopping
        if val_acc >= 99.0:  # More realistic threshold
            print(f"Early stopping at epoch {epoch+1}")
            break
        
        # Clear CUDA cache
        if DEVICE == "cuda":
            torch.cuda.empty_cache()
    
    # Plot results
    plot_training_curves(train_losses, train_accs, val_losses, val_accs)
    
    # Save model
    MODEL_PATH = Path("models")
    MODEL_PATH.mkdir(parents=True, exist_ok=True)
    torch.save(model.state_dict(), MODEL_PATH / "knot_detector.pth")
    print(f"Model saved to {MODEL_PATH / 'knot_detector.pth'}")

Model: 196,795,038 parameters, 751.08 MB
Starting training...

Epoch 1/20


RuntimeError: Given groups=1, weight of size [4, 1, 11, 11], expected input[32, 3, 480, 480] to have 1 channels, but got 3 channels instead

## MAJOR ISSUES IDENTIFIED:

REDUNDANCIES FOUND:

1. **Data Loading**: Multiple data loader definitions with inconsistent variable names
   - train_dataloader vs train_loader
   - Multiple DataLoader creations with same parameters

2. **Helper Function Downloads**: Duplicate download attempts with slightly different code

3. **Class Remapping**: Multiple similar functions doing the same thing

4. **Training Functions**: train_step and test_step have similar code but different interfaces

5. **Variable Names**: Inconsistent naming throughout (train_dataloader vs train_loader)

6. **Model Evaluation**: Multiple evaluation functions doing similar things

7. **Training Loops**: Two separate training loops with nearly identical code

8. **Import Statements**: Scattered imports instead of consolidated at top

QUESTIONS/SUGGESTIONS:

1. **Model Architecture**: Your FC layers are very large (4096->4096). Consider:
   - Global Average Pooling instead of large FC layers
   - Smaller hidden dimensions
   - This would reduce parameters significantly

2. **Data Splitting**: You're using test set for validation. Consider:
   - Creating a proper train/val/test split
   - Keeping test set untouched until final evaluation

3. **Loss Function**: MSE for classification seems unusual. Consider:
   - CrossEntropyLoss for multi-class classification
   - BCELoss for binary classification

4. **Early Stopping**: 99.9% accuracy threshold is very aggressive and may cause overfitting

5. **Code Organization**: Consider breaking this into separate modules:
   - data_utils.py
   - model.py
   - train_utils.py
   - evaluate.py
